In [104]:
import pandas as pd
import numpy as np

In [105]:
df=pd.read_csv("Churn_Modelling.csv")

In [106]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [107]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [108]:
df.drop(columns=["Surname","RowNumber","CustomerId"],inplace=True)

In [109]:
df.shape

(10000, 11)

In [110]:
df.drop_duplicates(inplace=True)

In [111]:
df["Gender"]=df.Gender.map({"Female":1,"Male":0})

In [112]:
df.Geography.unique()

array(['France', 'Spain', 'Germany'], dtype=object)

In [113]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

In [114]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False,drop='first')

In [115]:
x=df.drop(columns="Exited")
y=df.Exited

In [116]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [117]:
x_train.shape

(8000, 10)

In [118]:
y_train.shape

(8000,)

In [119]:
x_train_ohe=ohe.fit_transform(x_train[["Geography"]])
x_test_ohe=ohe.transform(x_test[["Geography"]])

In [120]:
feature=ohe.get_feature_names_out()

In [121]:
x_train_ohe=pd.DataFrame(x_train_ohe,columns=feature)
x_test_ohe=pd.DataFrame(x_test_ohe,columns=feature)

In [122]:
x_train=x_train.reset_index(drop=True)
x_test=x_test.reset_index(drop=True)

In [123]:
x_train=x_train.drop(columns="Geography")
x_test=x_test.drop(columns="Geography")

In [124]:
x_train=pd.concat([x_train_ohe,x_train],axis=1)
x_test=pd.concat([x_test_ohe,x_test],axis=1)    

In [125]:
x_train.shape

(8000, 11)

In [126]:
x_test

,Geography_Germany,Geography_Spain,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,1.0,0.0,733,1,33,7,187257.94,1,0,1,190430.81
1,0.0,0.0,629,1,37,10,99546.25,3,0,1,25136.95
2,1.0,0.0,617,0,44,9,49157.09,2,1,0,53294.17
3,0.0,1.0,794,1,30,1,154970.54,1,0,1,156768.45
4,0.0,0.0,598,0,48,6,120682.53,1,1,0,30635.52
...,...,...,...,...,...,...,...,...,...,...,...
1995,0.0,0.0,818,1,49,2,0.00,1,0,1,192298.84
1996,0.0,0.0,699,0,27,1,0.00,2,1,0,93003.21
1997,0.0,1.0,721,0,33,5,0.00,2,0,1,117626.90
1998,0.0,0.0,814,0,49,8,0.00,2,0,0,157822.54


In [127]:
model=Sequential([
    keras.layers.Dense(50,activation="relu",input_shape=(11,)),
    keras.layers.Dense(25,activation="relu"),
    keras.layers.Dense(1,activation="sigmoid")
])

c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [128]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (None, 50)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,901 (7.43 KB)

 Trainable params: 1,901 (7.43 KB)

 Non-trainable params: 0 (0.00 B)

In [129]:
model.compile(metrics=["accuracy"],loss="binary_crossentropy",optimizer="Adam")

In [131]:
model.fit(epochs=20,x=x_train,y=y_train)

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 811us/step - accuracy: 0.6821 - loss: 260.1435
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 723us/step - accuracy: 0.6858 - loss: 101.3246
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 732us/step - accuracy: 0.6844 - loss: 94.3905
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 719us/step - accuracy: 0.6871 - loss: 115.5622
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 729us/step - accuracy: 0.6931 - loss: 97.7615 
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 731us/step - accuracy: 0.6829 - loss: 81.1496
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 713us/step - accuracy: 0.6789 - loss: 75.6736
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 756us/step - accuracy: 0.6869 - loss: 83.3757
Epoch 9/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 731us/step - accuracy: 0.6926 - loss: 75.7526
Epoch 10/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 749us/step - accuracy: 0.6826 - loss: 74.2894
Epoch 11/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 729us/step - accuracy: 0.6908 - loss: 75.8263
Epoch 12/20
250

In [132]:
y_pred=model.predict(x_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


In [137]:
loss,accuracy=model.evaluate(x_test,y_test)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 778us/step - accuracy: 0.7755 - loss: 37.3830


In [139]:
print(accuracy)

0.7754999995231628
